<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 03 · Building a Market Data Pipeline with EODHD

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks/labs"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Reading Credentials Without Hard-Coding Them
The first design decision is to keep the API key out of version control.


In [ ]:
from labs.lab03_eodhd_pipeline import EODHDClient

In [ ]:
try:
    client = EODHDClient.from_creds()
except FileNotFoundError:
    client = None

client

In [ ]:
(
    client.base_url
    if client is not None
    else "credentials unavailable; using stored sample data"
)


## Wrapping the EOD Endpoint
The core retrieval logic lives as a method on the EODHDClient class.


## Downloading One Daily Price Series
Once the client exists, pulling a single history is straightforward.


In [ ]:
from labs.lab03_eodhd_pipeline import load_sample_dataset

if client is not None:
    history = client.get_eod(
        "AAPL",
        start="2024-01-02",
        stop="2025-03-31",
    )
else:
    sample = load_sample_dataset()
    history = (
        sample.loc[sample["symbol"] == "AAPL"]
        .set_index("date")
        .sort_index()
    )

history.tail()

In [ ]:
history[["close", "adjusted_close", "volume"]].head().round(2)

## Building a Multi-Symbol Market Panel
A single series is useful, but most research needs a panel.


In [ ]:
from labs.lab03_eodhd_pipeline import load_sample_dataset

In [ ]:
panel = load_sample_dataset()

In [ ]:
panel.head().round(2)

## Pivoting the Panel into Price and Return Views
Long-form tables are convenient for storage and joins, but many analytics are
easier in a wide matrix.


In [ ]:
from labs.lab03_eodhd_pipeline import close_matrix

In [ ]:
from labs.lab03_eodhd_pipeline import returns_matrix

In [ ]:
prices = close_matrix(panel)

In [ ]:
returns = returns_matrix(panel)

In [ ]:
prices.head(3).round(2)

In [ ]:
returns.corr().round(3)

## Figure Generation (Optional)
Run the lab figure scripts under `code/figures/` to regenerate the PNG files
under `assets/figures/`.


In [ ]:
# Figure generation code adapted from `code/figures/lab03_normalized_prices.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from labs.lab03_eodhd_pipeline import DEFAULT_SYMBOLS
from labs.lab03_eodhd_pipeline import close_matrix
from labs.lab03_eodhd_pipeline import load_sample_dataset

SYMBOLS = DEFAULT_SYMBOLS

def main() -> None:
    """Generate the normalized adjusted-close comparison figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    panel = load_sample_dataset()
    prices = close_matrix(panel)[list(SYMBOLS)]
    normalized = prices / prices.iloc[0] * 100.0

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for symbol in SYMBOLS:
        ax.plot(normalized.index, normalized[symbol], label=symbol)
    ax.set_title("Normalized adjusted closes from the stored sample")
    ax.set_ylabel("Index level")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(ncol=2)

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab03_return_correlation.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from labs.lab03_eodhd_pipeline import DEFAULT_SYMBOLS
from labs.lab03_eodhd_pipeline import load_sample_dataset
from labs.lab03_eodhd_pipeline import returns_matrix

SYMBOLS = DEFAULT_SYMBOLS

def main() -> None:
    """Generate a compact lower-triangle correlation figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update(
        {
            "font.family": "serif",
            "font.size": 7,
            "figure.dpi": 300,
        }
    )

    panel = load_sample_dataset()
    corr = returns_matrix(panel)[list(SYMBOLS)].corr()
    fig = plt.figure(figsize=(3.1, 2.6))
    grid = fig.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[1.0, 0.055],
        left=0.18,
        right=0.86,
        bottom=0.16,
        top=0.92,
        wspace=0.18,
    )
    ax = fig.add_subplot(grid[0, 0])
    cax = fig.add_subplot(grid[0, 1])
    cmap = mpl.cm.viridis
    norm = mpl.colors.Normalize(vmin=0.2, vmax=1.0)

    n = len(SYMBOLS)
    for row in range(n):
        for col in range(n):
            if col > row:
                continue
            value = float(corr.iloc[row, col])
            color = cmap(norm(value))
            radius = 0.40 if row == col else 0.34
            circle = plt.Circle((col, row), radius, color=color, ec="none")
            ax.add_patch(circle)
            text_color = "black" if value > 0.82 else "white"
            ax.text(
                col,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=5.5,
            )

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.set_xticks(range(n))
    ax.set_xticklabels(SYMBOLS, fontsize=6)
    ax.set_yticks(range(n))
    ax.set_yticklabels(SYMBOLS, fontsize=6)
    ax.tick_params(length=0, pad=2)

    for row in range(n + 1):
        ax.axhline(row - 0.5, color="0.88", lw=0.6, zorder=0)
        ax.axvline(row - 0.5, color="0.88", lw=0.6, zorder=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(
        sm,
        cax=cax,
        ticks=[0.2, 0.5, 0.8, 1.0],
    )
    cbar.ax.tick_params(labelsize=5.5, length=0)
    cbar.outline.set_linewidth(0.4)


main()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
